In [22]:
from pathlib import Path
import sys
import pandas as pd
import yaml

sys.path.append(str(Path.cwd().parents[3]))
from utils.helpers import save_csv_file, load_data

# Functions
#### below you can find functions that are executed automatically via calling the main function ***data_cleaning_main(config_path, data_file_path)***

### Load config

In [23]:
def load_config(config_path):
    """
    Load configuration from YAML file
    
    Args:
        config_path (str): Path to the YAML configuration file
    
    Returns:
        dict: Configuration dictionary
    """
    try:
        with open(config_path, 'r', encoding='utf-8') as file:
            config = yaml.safe_load(file)
        print(f"Configuration loaded successfully from {config_path}")
        return config
    except Exception as e:
        print(f"Error loading configuration: {e}")
        return None

### Remove columns

In [24]:
def remove_columns(df, columns_to_remove):
    """
    Remove specified columns from DataFrame
    
    Args:
        df (pandas.DataFrame): Input DataFrame
        columns_to_remove (list): List of column names to remove
    
    Returns:
        pandas.DataFrame: DataFrame with columns removed
    """
    existing_columns = [col for col in columns_to_remove if col in df.columns]
    missing_columns = [col for col in columns_to_remove if col not in df.columns]
    
    if missing_columns:
        print(f"Warning: These columns were not found: {missing_columns}")
    
    if existing_columns:
        df_cleaned = df.drop(columns=existing_columns)
        print(f"Removed {len(existing_columns)} columns")
    else:
        df_cleaned = df.copy()
        print("No columns to remove")
    
    return df_cleaned

### Filling NaN values (students' responses)

In [25]:
def fill_missing_response_values(df, target_column, source_column):
    """
    Fill missing values in target column with values from source column
    
    Args:
        df (pandas.DataFrame): Input DataFrame
        target_column (str): Column to fill missing values
        source_column (str): Column to use as source for filling
    
    Returns:
        pandas.DataFrame: DataFrame with filled values
    """
    if target_column not in df.columns:
        print(f"Target column '{target_column}' not found - skipping fill operation")
        return df
    
    if source_column not in df.columns:
        print(f"Source column '{source_column}' not found - skipping fill operation")
        return df
    
    df_filled = df.copy()
    missing_count = df_filled[target_column].isna().sum()
    
    if missing_count > 0:
        df_filled[target_column] = df_filled[target_column].fillna(df_filled[source_column])
        print(f"Filled {missing_count} missing values in '{target_column}'")
    else:
        print(f"No missing values found in '{target_column}'")
    
    return df_filled


# After filling missing values from source to target column, there are still NaN values in the DataFrame.
# We will fill them with a default values "No Response"
# And count how many NaN values were filled
def fill_NaN_values(df, fill_value="No Response"):
    """
    Fill NaN values in DataFrame with a specified value
    
    Args:
        df (pandas.DataFrame): Input DataFrame
        fill_value (str): Value to fill NaN entries with
    
    Returns:
        pandas.DataFrame: DataFrame with NaN values filled
    """
    df_filled = df.copy()
    total_nan_before = df.isna().sum().sum()
    
    if total_nan_before > 0:
        df_filled = df_filled.fillna(fill_value)
        print(f"Filled {total_nan_before} NaN values with '{fill_value}'")
    else:
        print("No NaN values found to fill")
    
    return df_filled

### Rename columns

In [26]:
def rename_columns(df, column_rename_map):
    """
    Rename columns according to the provided mapping
    
    Args:
        df (pandas.DataFrame): Input DataFrame
        column_rename_map (dict): Dictionary mapping old names to new names
    
    Returns:
        pandas.DataFrame: DataFrame with renamed columns
    """
    existing_columns = {old: new for old, new in column_rename_map.items() if old in df.columns}
    missing_columns = {old: new for old, new in column_rename_map.items() if old not in df.columns}
    
    if missing_columns:
        print(f"Warning: These columns were not found for renaming: {list(missing_columns.keys())}")
    
    if existing_columns:
        df_renamed = df.rename(columns=existing_columns)
        print(f"Renamed {len(existing_columns)} columns")
    else:
        df_renamed = df.copy()
        print("No columns to rename")
    
    return df_renamed

### Convert columns with object type to datetime type

In [27]:
def convert_datetime_columns(df, datetime_columns):
    """
    Convert specified columns to datetime format and remove timezone info
    
    Args:
        df (pandas.DataFrame): Input DataFrame
        datetime_columns (list): List of column names to convert to datetime
    
    Returns:
        pandas.DataFrame: DataFrame with datetime columns converted
    """
    df_converted = df.copy()
    
    # Filter columns to only those that exist in the DataFrame
    existing_datetime_columns = [col for col in datetime_columns if col in df_converted.columns]
    missing_datetime_columns = [col for col in datetime_columns if col not in df_converted.columns]
    
    if missing_datetime_columns:
        print(f"Warning: These datetime columns were not found: {missing_datetime_columns}")
    
    for col in existing_datetime_columns:
        try:
            # Convert to datetime
            df_converted[col] = pd.to_datetime(df_converted[col])
            # Remove timezone info if present
            if df_converted[col].dt.tz is not None:
                df_converted[col] = df_converted[col].dt.tz_localize(None)
            print(f"Converted '{col}' to datetime (timezone removed)")
        except Exception as e:
            print(f"Error converting '{col}' to datetime: {e}")
    
    return df_converted

### Convert columns with object type to string type

In [28]:
def convert_string_columns(df, string_columns):
    """
    Convert specified columns to string type
    
    Args:
        df (pandas.DataFrame): Input DataFrame
        string_columns (list): List of column names to convert to string
    
    Returns:
        pandas.DataFrame: DataFrame with string columns converted
    """
    df_converted = df.copy()
    
    # Filter columns to only those that exist in the DataFrame
    existing_string_columns = [col for col in string_columns if col in df_converted.columns]
    missing_string_columns = [col for col in string_columns if col not in df_converted.columns]
    
    if missing_string_columns:
        print(f"Warning: These string columns were not found: {missing_string_columns}")
    
    for col in existing_string_columns:
        try:
            df_converted[col] = df_converted[col].astype(str)
            print(f"Converted '{col}' to string")
        except Exception as e:
            print(f"Error converting '{col}' to string: {e}")
    
    return df_converted

### Main function 
* Execute the pre-defined cleaning process. 
* Take two arguments config and file_path.

In [29]:
def clean_data_pipeline(config, file_path):
    """
    Complete data cleaning pipeline
    
    Args:
        config (dict): Configuration dictionary with all parameters
        file_path (str): Path to the data file
    
    Returns:
        pandas.DataFrame: Cleaned DataFrame
    """
    print("Starting data cleaning pipeline...")
    print("=" * 50)

    # Load data
    df = load_data(file_path)
    if df is None:
        return None
    
    # Fill missing values if configured
    if 'fill_missing_columns' in config:
        df = fill_missing_response_values(
            df, 
            config['fill_missing_columns']['target'],
            config['fill_missing_columns']['source']
        )

    # Fill NaN values with a default value
    df = fill_NaN_values(df, fill_value="No Response")
    
    # Convert datetime columns (before renaming, using original column names)
    if 'datetime_columns' in config:
        df = convert_datetime_columns(df, config['datetime_columns'])

    # Convert string columns
    if 'string_columns' in config:
        df = convert_string_columns(df, config['string_columns'])
    
    # Remove specified columns
    if 'columns_to_remove' in config:
        df = remove_columns(df, config['columns_to_remove'])
    
    # Rename columns
    if 'column_rename_map' in config:
        df = rename_columns(df, config['column_rename_map'])

    
    print("=" * 50)
    print(f"Data cleaning completed. Final shape: {df.shape}")
    
    return df

## Main data cleaning function

In [30]:
def data_cleaning_main(config_path, data_file_path):
    """
    Main function to execute the data cleaning pipeline
    """
    # Load configuration from YAML file
    config_path =config_path
    config = load_config(config_path)

    if config:
        # Set the data file path (can override the one in config)
        data_file_path = data_file_path
        
        # Run the cleaning pipeline
        cleaned_df = clean_data_pipeline(config, data_file_path)
        
    else:
        print("Failed to load configuration file")
    return cleaned_df

## Usage of the data_cleaning_main(config_path, data_file_path)

In [31]:
cleaned_df = data_cleaning_main('../../../../configs/dataset_config.yaml', '../data/csv_files//IN2120_2024_anon.csv')

Configuration loaded successfully from ../../../../configs/dataset_config.yaml
Starting data cleaning pipeline...
Data loaded successfully from ../data/csv_files//IN2120_2024_anon.csv
Shape: (47565, 30)
Filled 31797 missing values in 'Column1.result.ext_inspera_questions.ext_inspera_candidateResponses.ext_insper.1'
Filled 21405 NaN values with 'No Response'
Converted 'Column1.result.ext_inspera_startTime' to datetime (timezone removed)
Converted 'Column1.result.ext_inspera_endTime' to datetime (timezone removed)
Converted 'Column1.result.ext_inspera_questions.ext_inspera_candidateResponses.ext_inspera_' to string
Converted 'Column1.result.ext_inspera_questions.ext_inspera_candidateResponses.ext_insper.1' to string
Converted 'Column1.result.ext_inspera_questions.ext_inspera_questionTitle' to string
Removed 14 columns
Renamed 16 columns
Data cleaning completed. Final shape: (47565, 16)


In [32]:
save_csv_file(cleaned_df, 'IN2120_2024_cleaned_data', '../data/cleaned_data/')

✅ DataFrame saved as: ../data/cleaned_data/IN2120_2024_cleaned_data.csv
